In [3]:
orders = spark.read.table("SL_Orders")

customers = spark.read.table("SL_Customers")

payments = spark.read.table("SL_Order_Payments")

reviews = spark.read.table("SL_Order_Reviews")

categories = spark.read.table("SL_Product_Category")

StatementMeta(, 88403063-ca63-4f41-96be-d65937c201a2, 5, Finished, Available, Finished, False)

Create ****DIM_Customers***


In [5]:
from pyspark.sql.functions import *

dim_customers = (
    customers
    .select(
        "customer_id",
        "customer_unique_id",
        "customer_city",
        "customer_state"
    )
    .dropDuplicates(["customer_id"])
)

StatementMeta(, 88403063-ca63-4f41-96be-d65937c201a2, 7, Finished, Available, Finished, False)

In [6]:
dim_customers.write \
.mode("overwrite") \
.saveAsTable("Dim_Customers")

StatementMeta(, 88403063-ca63-4f41-96be-d65937c201a2, 8, Finished, Available, Finished, False)

****DIm_product_Category****

In [7]:
dim_category = (
    categories
    .select(
        "product_category_name",
        "product_category_name_english"
    )
    .dropDuplicates()
)

StatementMeta(, 88403063-ca63-4f41-96be-d65937c201a2, 9, Finished, Available, Finished, False)

In [8]:
dim_category.write \
.mode("overwrite") \
.saveAsTable("Dim_Product_Category")

StatementMeta(, 88403063-ca63-4f41-96be-d65937c201a2, 10, Finished, Available, Finished, False)

****Create Dim_Date****

In [9]:
from pyspark.sql.functions import *

dim_date = (
    orders
    .select("order_purchase_timestamp")
    .dropDuplicates()
    .withColumnRenamed(
        "order_purchase_timestamp",
        "date"
    )
    .withColumn("year", year("date"))
    .withColumn("month", month("date"))
    .withColumn("month_name", date_format("date", "MMMM"))
    .withColumn("quarter", quarter("date"))
    .withColumn("day", dayofmonth("date"))
    .withColumn("week", weekofyear("date"))
)

StatementMeta(, 88403063-ca63-4f41-96be-d65937c201a2, 11, Finished, Available, Finished, False)

In [10]:
dim_date.write \
.mode("overwrite") \
.saveAsTable("Dim_Date")

StatementMeta(, 88403063-ca63-4f41-96be-d65937c201a2, 12, Finished, Available, Finished, False)

***Create Fact_Order_Sales***

In [11]:
fact_sales = (
    orders.alias("o")
    .join(
        payments.alias("p"),
        "order_id",
        "left"
    )
    .join(
        reviews.alias("r"),
        "order_id",
        "left"
    )
)

StatementMeta(, 88403063-ca63-4f41-96be-d65937c201a2, 13, Finished, Available, Finished, False)

In [12]:
fact_sales = fact_sales.select(
    "order_id",
    "customer_id",
    "order_purchase_timestamp",
    "order_status",
    "payment_value",
    "payment_installments",
    "review_score"
)

StatementMeta(, 88403063-ca63-4f41-96be-d65937c201a2, 14, Finished, Available, Finished, False)

In [13]:
fact_sales.write \
.mode("overwrite") \
.saveAsTable("Fact_Order_Sales")

StatementMeta(, 88403063-ca63-4f41-96be-d65937c201a2, 15, Finished, Available, Finished, False)

***Business KPIs***
***1. Total Revenue***

In [14]:
from pyspark.sql.functions import sum

fact_sales.select(
    sum("payment_value").alias("Total_Revenue")
).show()

StatementMeta(, 88403063-ca63-4f41-96be-d65937c201a2, 16, Finished, Available, Finished, False)

+-------------------+
|      Total_Revenue|
+-------------------+
|1.604176217000004E7|
+-------------------+



***2. Average Order Value***

In [15]:
from pyspark.sql.functions import avg

fact_sales.select(
    avg("payment_value").alias("Average_Order_Value")
).show()

StatementMeta(, 88403063-ca63-4f41-96be-d65937c201a2, 17, Finished, Available, Finished, False)

+-------------------+
|Average_Order_Value|
+-------------------+
|  154.0314766769731|
+-------------------+



***3. Total Orders***

In [16]:
fact_sales.select(
    countDistinct("order_id").alias("Total_Orders")
).show()

StatementMeta(, 88403063-ca63-4f41-96be-d65937c201a2, 18, Finished, Available, Finished, False)

+------------+
|Total_Orders|
+------------+
|       99441|
+------------+



***4. Orders by Status***

In [17]:
fact_sales.groupBy(
    "order_status"
).count().show()

StatementMeta(, 88403063-ca63-4f41-96be-d65937c201a2, 19, Finished, Available, Finished, False)

+------------+------+
|order_status| count|
+------------+------+
|     shipped|  1169|
|    canceled|   664|
|    invoiced|   325|
|     created|     5|
|   delivered|101016|
| unavailable|   649|
|  processing|   320|
|    approved|     2|
+------------+------+



***5. Monthly Sales***

In [18]:
from pyspark.sql.functions import year, month, sum

monthly_sales = (
    fact_sales
    .groupBy(
        year("order_purchase_timestamp").alias("Year"),
        month("order_purchase_timestamp").alias("Month")
    )
    .agg(
        sum("payment_value").alias("Revenue")
    )
    .orderBy("Year", "Month")
)

monthly_sales.show()

StatementMeta(, 88403063-ca63-4f41-96be-d65937c201a2, 21, Finished, Available, Finished, False)

+----+-----+------------------+
|Year|Month|           Revenue|
+----+-----+------------------+
|2016|    9|            252.24|
|2016|   10|59219.509999999995|
|2016|   12|             19.62|
|2017|    1|138786.60000000003|
|2017|    2| 292806.1899999998|
|2017|    3| 450671.9999999997|
|2017|    4|418445.07999999996|
|2017|    5| 594986.1699999997|
|2017|    6| 513169.3099999997|
|2017|    7| 593924.7799999996|
|2017|    8| 677273.5199999997|
|2017|    9| 729966.2299999997|
|2017|   10| 781846.7899999993|
|2017|   11|1198342.0299999998|
|2017|   12| 880110.7699999996|
|2018|    1|1116634.9599999988|
|2018|    2|  996933.249999999|
|2018|    3| 1161879.480000001|
|2018|    4|        1162364.53|
|2018|    5|1154156.0400000003|
+----+-----+------------------+
only showing top 20 rows



***6. Review Score Analysis***

In [19]:
from pyspark.sql.functions import avg

fact_sales.groupBy(
    "review_score"
).agg(
    avg("payment_value").alias("Average_Order_Value")
).show()

StatementMeta(, 88403063-ca63-4f41-96be-d65937c201a2, 23, Finished, Available, Finished, False)

+------------+-------------------+
|review_score|Average_Order_Value|
+------------+-------------------+
|        NULL| 168.44914781297135|
|           1| 186.38681529740236|
|           3| 145.31800141911063|
|           5| 149.92771693188502|
|           4|  148.2385397896012|
|           2| 163.42887033100507|
+------------+-------------------+

